# Open Judy — Ground-Truth Audio-Subtitle Synchronization Benchmark & Viterbi DP Acoustic Aligner Evaluation

## 1. Benchmark Dataset & Methodology
To rigorously evaluate and optimize audio-subtitle timestamp synchronization in the Open Judy transcription pipeline (`pipelines/transcription-pipeline/gemini_client.py` and `db.py`), we downloaded real spoken Portuguese utterances and verbatim transcripts from **`facebook/multilingual_librispeech` (`portuguese`, `test` split)** via HuggingFace (`notebooks/datasets/ground_truth_sync/`) and constructed **4 multi-minute conversational streams** (`36` utterances, `696.9s` / `11.6 minutes` total audio) with exact millisecond ground-truth boundaries (`[gtStartTimeMs, gtEndTimeMs]`):

1. **`mls_pt_stream_01_clean46dB.wav`** (`182.62s`, `-46.0 dBFS` clean studio floor, `9` segments)
2. **`mls_pt_stream_02_noisy28dB.wav`** (`156.76s`, `-28.5 dBFS` simulated courtroom HVAC noise floor, `9` segments)
3. **`mls_pt_stream_03_clean46dB.wav`** (`189.20s`, `-46.0 dBFS` clean studio floor, `9` segments)
4. **`mls_pt_stream_04_noisy28dB.wav`** (`168.31s`, `-28.5 dBFS` simulated courtroom HVAC noise floor, `9` segments)

We also evaluate on the real courtroom hearing **`00 - video 1min .mp4`** (`7c2812b0-1aa3-46f0-ad9f-d9175c5bb153`, `88.39s`, `16` segments) with human-verified acoustic ground-truth timestamps.

### Root Causes Identified & Fixed
- **Root Cause 1 (Fixed `-35dB` `silencedetect` Blindness on Courtroom Audio)**: On courtroom recordings with `-28dB` to `-30dB` HVAC background noise (`mls_pt_stream_02_noisy28dB`, `mls_pt_stream_04_noisy28dB`, and `00 - video 2h 194MB .mp4`), `ffmpeg silencedetect=noise=-35dB:d=0.35` detected **only 1 single island across the entire clip (0 pauses)**, forcing the legacy pipeline to fall back to uniform linear character interpolation with `100ms` artificial gaps.
- **Root Cause 2 (Short `<350ms` Inter-Sentence Pause Blindness)**: Even on clean audio (`00 - video 1min .mp4`), `d=0.35` missed `220ms–340ms` pauses (`7.32s–7.60s`, `28.14s–28.42s`, `80.50s–82.16s`), placing `Seg 15 (AUTOR: "Sim.")` inside silence (`80600–81339ms` instead of `82160–82800ms`).
- **Root Cause 3 (Upgraded Adaptive 20ms RMS Energy Envelope + Viterbi DP Aligner)**: We replaced fixed-threshold `silencedetect` + linear interpolation with `_extract_rms_energy_profile` (computing the 15th-percentile noise floor `p15` and 90th-percentile speech peak `p90` at `20ms` frame resolution, setting an adaptive threshold `thresh = max(p15 + 4.2, min(p90 - 8.5, p15 + 0.35 * (p90 - p15)))`, and splitting long speech islands at `>= 4.5dB` local energy valleys) paired with `align_segments_viterbi_acoustic` (a Viterbi Dynamic Programming aligner using Portuguese vowel-nucleus syllable weights, utterance span regularization, gap quietness scoring, and locally-gated Huber priors).

In [1]:
import json
import os

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
report_path = os.path.join(REPO_ROOT, "notebooks", "outputs", "audio_sync_ground_truth_eval_report.json")

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

ov = report["overallSummary"]
print("=== OVERALL BENCHMARK SUMMARY (36 Ground-Truth Segments across 4 Multi-Minute Streams) ===")
header = f"{'Pipeline Variant':<42} | {'Start MAE':>9} | {'Start Med':>9} | {'End MAE':>9} | {'End Med':>9} | {'IoU (%)':>7} | {'<=250ms':>7} | {'<=500ms':>7} | {'<=1000ms':>8}"
print(header)
print("-" * len(header))
for label, key in [
    ("1. Legacy Baseline (-35dB + Linear)", "legacyBaseline"),
    ("2. Raw Gemini 3.8 Flash (Unaligned LLM)", "rawGemini38Flash"),
    ("3. Upgraded Pipeline (Blind Viterbi DP)", "upgradedBlindViterbiDp"),
    ("4. Upgraded Pipeline (Full Gemini+Viterbi)", "upgradedFullPipeline"),
]:
    m = ov[key]
    print(
        f"{label:<42} | {m['startMaeMs']:7.1f}ms | {m['startMedianErrMs']:7.1f}ms | "
        f"{m['endMaeMs']:7.1f}ms | {m['endMedianErrMs']:7.1f}ms | {m['meanIouPct']:6.1f}% | "
        f"{m['within250msPct']:6.1f}% | {m['within500msPct']:6.1f}% | {m['within1000msPct']:7.1f}%"
    )


=== OVERALL BENCHMARK SUMMARY (36 Ground-Truth Segments across 4 Multi-Minute Streams) ===
Pipeline Variant                           | Start MAE | Start Med |   End MAE |   End Med | IoU (%) | <=250ms | <=500ms | <=1000ms
-----------------------------------------------------------------------------------------------------------------------------------
1. Legacy Baseline (-35dB + Linear)        |  1862.1ms |  1458.5ms |  1748.6ms |  1464.0ms |   82.0% |   15.3% |   25.0% |    37.5%
2. Raw Gemini 3.8 Flash (Unaligned LLM)    |  3935.8ms |  2620.0ms |  3584.2ms |  2190.0ms |   65.7% |    2.8% |    8.3% |    15.3%
3. Upgraded Pipeline (Blind Viterbi DP)    |    58.3ms |    10.0ms |   186.4ms |    20.0ms |   98.6% |   86.1% |   93.1% |    95.8%
4. Upgraded Pipeline (Full Gemini+Viterbi) |    96.1ms |    10.0ms |   223.1ms |    25.0ms |   98.2% |   83.3% |   90.3% |    93.1%


In [2]:
print("=== PER-STREAM BREAKDOWN (Clean -46.0dB Studio vs. Noisy -28.5dB Courtroom HVAC) ===")
for sr in report["streamReports"]:
    print(
        f"\nStream: {sr['streamId']} ({sr['durationSeconds']}s, Noise Floor: {sr['backgroundNoiseFloorDb']}dB, "
        f"Fixed -35dB Islands: {sr['fixed35dBDetectedIslands']}, Adaptive 20ms RMS Islands: {sr['adaptiveRmsDetectedIslands']} @ {sr['adaptiveThresholdDb']}dB)"
    )
    for name, m in [
        ("Legacy Baseline", sr["baselineMetrics"]),
        ("Raw Gemini 3.8 Flash", sr["rawGeminiMetrics"]),
        ("Upgraded Blind Viterbi DP", sr["upgradedBlindViterbiMetrics"]),
        ("Upgraded Full Pipeline", sr["upgradedFullPipelineMetrics"]),
    ]:
        print(
            f"  {name:<26}: Start MAE={m['startMaeMs']:6.1f}ms (Med={m['startMedianErrMs']:5.1f}ms) | "
            f"End MAE={m['endMaeMs']:6.1f}ms (Med={m['endMedianErrMs']:5.1f}ms) | "
            f"IoU={m['meanIouPct']:5.1f}% | <=250ms={m['within250msPct']:5.1f}% | <=500ms={m['within500msPct']:5.1f}%"
        )


=== PER-STREAM BREAKDOWN (Clean -46.0dB Studio vs. Noisy -28.5dB Courtroom HVAC) ===

Stream: mls_pt_stream_01_clean46dB (182.62s, Noise Floor: -46.0dB, Fixed -35dB Islands: 50, Adaptive 20ms RMS Islands: 88 @ -43.74dB)
  Legacy Baseline           : Start MAE=1539.3ms (Med=1253.0ms) | End MAE=1240.1ms (Med=841.0ms) | IoU= 86.8% | <=250ms= 11.1% | <=500ms= 27.8%
  Raw Gemini 3.8 Flash      : Start MAE=3620.0ms (Med=1850.0ms) | End MAE=3295.6ms (Med=1580.0ms) | IoU= 70.1% | <=250ms=  0.0% | <=500ms=  5.6%
  Upgraded Blind Viterbi DP : Start MAE=   6.7ms (Med= 10.0ms) | End MAE= 211.1ms (Med= 10.0ms) | IoU= 98.8% | <=250ms= 94.4% | <=500ms= 94.4%
  Upgraded Full Pipeline    : Start MAE= 157.8ms (Med= 10.0ms) | End MAE= 357.8ms (Med= 10.0ms) | IoU= 97.4% | <=250ms= 83.3% | <=500ms= 83.3%

Stream: mls_pt_stream_02_noisy28dB (156.76s, Noise Floor: -28.5dB, Fixed -35dB Islands: 1, Adaptive 20ms RMS Islands: 116 @ -33.71dB)
  Legacy Baseline           : Start MAE=1415.4ms (Med=1173.0ms) | End 

In [3]:
c_eval = report.get("courtroom1MinEvaluation", {})
if c_eval:
    print("=== REAL COURTROOM HEARING EVALUATION: `00 - video 1min .mp4` (7c2812b0, 16 Segments) ===")
    p_m = c_eval["previousPipelineMetrics"]
    u_m = c_eval["upgradedViterbiMetrics"]
    print(f"Previous Pipeline : Start MAE={p_m['startMaeMs']:6.1f}ms (Med={p_m['startMedianErrMs']:5.1f}ms) | End MAE={p_m['endMaeMs']:6.1f}ms (Med={p_m['endMedianErrMs']:5.1f}ms) | IoU={p_m['meanIouPct']:5.1f}% | <=250ms={p_m['within250msPct']:5.1f}% | <=500ms={p_m['within500msPct']:5.1f}%")
    print(f"Upgraded Viterbi  : Start MAE={u_m['startMaeMs']:6.1f}ms (Med={u_m['startMedianErrMs']:5.1f}ms) | End MAE={u_m['endMaeMs']:6.1f}ms (Med={u_m['endMedianErrMs']:5.1f}ms) | IoU={u_m['meanIouPct']:5.1f}% | <=250ms={u_m['within250msPct']:5.1f}% | <=500ms={u_m['within500msPct']:5.1f}%")
    print("\nSegment-by-Segment Comparison on `00 - video 1min .mp4`:")
    for ps, us in zip(p_m["segments"], u_m["segments"]):
        idx = ps["segmentIndex"]
        print(
            f"  Seg {idx:2d}: GT=({ps['gtStartTimeMs']:5d}, {ps['gtEndTimeMs']:5d}) | "
            f"Prev=({ps['predStartTimeMs']:5d}, {ps['predEndTimeMs']:5d}) [err={ps['startErrorMs']:+5d}ms, {ps['endErrorMs']:+5d}ms] | "
            f"Upgraded=({us['predStartTimeMs']:5d}, {us['predEndTimeMs']:5d}) [err={us['startErrorMs']:+4d}ms, {us['endErrorMs']:+4d}ms]"
        )


=== REAL COURTROOM HEARING EVALUATION: `00 - video 1min .mp4` (7c2812b0, 16 Segments) ===
Previous Pipeline : Start MAE= 687.3ms (Med=210.5ms) | End MAE= 548.4ms (Med=175.0ms) | IoU= 76.5% | <=250ms= 56.2% | <=500ms= 65.6%
Upgraded Viterbi  : Start MAE=  14.1ms (Med=  0.0ms) | End MAE=  23.6ms (Med=  0.0ms) | IoU= 99.0% | <=250ms=100.0% | <=500ms=100.0%

Segment-by-Segment Comparison on `00 - video 1min .mp4`:
  Seg  1: GT=( 2300,  7320) | Prev=( 2311,  7400) [err=  +11ms,   +80ms] | Upgraded=( 2300,  7320) [err=  +0ms,   +0ms]
  Seg  2: GT=( 7600, 15760) | Prev=( 7600, 13500) [err=   +0ms, -2260ms] | Upgraded=( 7600, 15760) [err=  +0ms,   +0ms]
  Seg  3: GT=(16640, 21540) | Prev=(13700, 21690) [err=-2940ms,  +150ms] | Upgraded=(16640, 21620) [err=  +0ms,  +80ms]
  Seg  4: GT=(22300, 26220) | Prev=(22326, 26269) [err=  +26ms,   +49ms] | Upgraded=(22300, 26220) [err=  +0ms,   +0ms]
  Seg  5: GT=(28420, 33800) | Prev=(28090, 34200) [err= -330ms,  +400ms] | Upgraded=(28420, 33800) [err=  